# VeloceAI — Colab GPU Engine
Run this notebook in Google Colab with a GPU runtime. It starts a temporary Real-ESRGAN API and Cloudflare quick tunnel. The tunnel URL changes each session; stop the runtime when finished.

In [ ]:
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install fastapi uvicorn python-multipart opencv-python-headless basicsr facexlib gfpgan
import os, subprocess, sys, time, urllib.request, pathlib
if not pathlib.Path('/content/Real-ESRGAN').exists():
    subprocess.run(['git','clone','https://github.com/xinntao/Real-ESRGAN.git','/content/Real-ESRGAN'],check=True)
%cd /content/Real-ESRGAN
!pip -q install -r requirements.txt
!python setup.py develop
print('Real-ESRGAN dependencies installed.')

In [ ]:
from google.colab import files
import pathlib, subprocess, time, urllib.request
REPO='https://raw.githubusercontent.com/rezeyuri-dot/veloceai/main/colab_server.py'
urllib.request.urlretrieve(REPO, '/content/colab_server.py')
subprocess.run(['python','-c', "from basicsr.archs.rrdbnet_arch import RRDBNet; print('BasicSR import OK')"],check=True)
server_log=open('/content/veloceai-server.log','w')
server=subprocess.Popen([sys.executable,'/content/colab_server.py'],stdout=server_log,stderr=subprocess.STDOUT)
for i in range(30):
    try:
        print(urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=2).read().decode()); break
    except Exception: time.sleep(2)
else: raise RuntimeError('VeloceAI API did not start. Check /content/veloceai-server.log')

In [ ]:
import os, subprocess, time, re, pathlib, urllib.request
cloudflared='/usr/local/bin/cloudflared'
if not pathlib.Path(cloudflared).exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',cloudflared)
    os.chmod(cloudflared,0o755)
tunnel_log=open('/content/veloceai-tunnel.log','w')
tunnel=subprocess.Popen([cloudflared,'tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
public_url=None
for i in range(60):
    time.sleep(2)
    text=pathlib.Path('/content/veloceai-tunnel.log').read_text(errors='ignore')
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
    if match: public_url=match.group(0); break
if not public_url: raise RuntimeError('Tunnel URL not found. See /content/veloceai-tunnel.log')
print('VeloceAI temporary API URL:',public_url)
print('Health check:',urllib.request.urlopen(public_url+'/health',timeout=20).read().decode())